### RAG agent with Web search feature

In [1]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10503.61it/s]


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
SERPER_API_KEY=os.getenv("SERPER_API_KEY")

os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
os.environ["GROQ_API_KEY"]= GROQ_API_KEY
os.environ["SERPER_API_KEY"]= SERPER_API_KEY


In [4]:

from langchain_groq import ChatGroq
llm=ChatGroq(model_name="llama-3.1-8b-instant")

In [5]:
loader=WebBaseLoader("https://docs.smith.langchain.com/overview")

docs=loader.load()

In [6]:
docs

[Document(metadata={'source': 'https://docs.smith.langchain.com/overview', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageDocumentation IndexFetch the complete documentation index at: https://docs.langchain.com/llms.txtUse this file to discover all available pages before explo

In [7]:
documents = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200
).split_documents(docs)

In [8]:
documents

[Document(metadata={'source': 'https://docs.smith.langchain.com/overview', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageDocumentation IndexFetch the complete documentation index at: https://docs.langchain.com/llms.txtUse this file to discover all available pages before explo

In [9]:
vector=FAISS.from_documents(documents=documents,embedding=embeddings)

In [10]:
### making retriever (converts my query into embeddings and does similarity search into vector store)

retriever=vector.as_retriever()

In [11]:
retriever.invoke("how to upload a dataset")[0]

Document(id='0851780e-9a39-43c9-a563-79843a2d58e6', metadata={'source': 'https://docs.smith.langchain.com/overview', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='\u200bMore ways to build\nFleetDesign and deploy AI agents visually without writing code.Build an agentPrompt engineeringIterate on prompts with built-in versioning and collaboration to ship improvements faster.Test your promptsLangSmith CLIQuery and manage traces, datasets, experiments, and more from the terminal.Use the CLIStudioUse a visual interface to design, test, and refine applications end-to-end.Develop with Studio\n\u200bInfrastructure\nPlatform setupUse LangSmith in managed cloud, in a self-hosted environment, or hybrid to match your infrastructure and compliance needs.Choose how to set up LangSmithSecurity & complianceLangSmith meets the highest standards of data security and privacy with HIPAA, SOC 2 Type 2, and GDPR compliance. Meet with our team to learn more or visit our Trust

In [12]:
## Creating tool of retriever so that it can used by llm

from langchain.tools import tool

@tool
def retrieve_blog_posts(query: str) -> str:
    """Search and return information about posts."""
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

retriever_tool = retrieve_blog_posts

In [13]:
### Creating web search tool

from langchain_community.tools.tavily_search import TavilySearchResults

In [14]:
search = TavilySearchResults()

/var/folders/qz/mw466yp17txddtx0y1qsb1q00000gn/T/ipykernel_3297/1244456812.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search = TavilySearchResults()


In [15]:

search.invoke("what is the weather in SF")

[{'title': 'Weather San Francisco in April 2026: Temperature & Climate',
  'url': 'https://en.climate-data.org/north-america/united-states-of-america/california/san-francisco-385/t/april-4/',
  'content': "In San Francisco, average temperatures in April are around 12.7°C | 54.9°F. The water is usually 11.3°C | 52.3°F in the first third of the month, rises to 11.2°C | 52.2°F by mid-month, and finishes at about 11.2°C | 52.2°F by month’s end.\n\nThroughout April, sunshine in San Francisco ranges from around 8 hours to 11 hours daily.\n30.04 is the brightest day with 11 hours of sun, while 11.04 records only 8 hours.\nThe average sunshine duration is 9 hours in the early days, 9 hours around mid-month, and 10 hours by month’s end.\nSunshine peaks between 21-30, with 10 hours per day.\nThe cloudiest 10-day period is from 1-10, with just 9 hours of sunshine each day. [...] In San Francisco, average temperatures in April are around 12.7°C | 54.9°F. The water is usually 11.3°C | 52.3°F in the

In [16]:
from langsmith import Client

client = Client()

prompt = client.pull_prompt("hwchase17/openai-functions-agent")

prompt.messages[0].prompt.template

'You are a helpful assistant'

In [17]:
from langchain.agents import create_agent

In [18]:
tools=[search,retriever_tool]

In [19]:
## Creating agent

agent=create_agent(
    model=llm,
    tools=tools,
    system_prompt=prompt.messages[0].prompt.template
)

In [20]:
response=agent.invoke({
    "messages":[
        {
            "role":"user",
            "content":"What is java, search with search tool"
        }
    ]
})

In [21]:
print(response["messages"][-1].content)

Based on the search results, Java is a high-level, general-purpose programming language that is designed to let programmers "write once, run anywhere" (WORA), meaning that compiled Java code can run on all platforms that support Java without the need to recompile. It is intended to be a platform-independent language, which means that Java code can run on any device that has a Java Virtual Machine (JVM) installed, regardless of the underlying computer architecture. Java is a versatile language that can be used for a wide range of applications, including Android development, web development, big data processing, and cloud-based microservices. It is known for its reliability, ease of use, and robust security features, making it a popular choice for software developers.


## ReAct Agent Design Pattern

ReAct is a framework where the agent alternates between reasoning and actions.

---

## Flow

Thought → reasoning step  
Action → call a tool  
Observation → get result  
Repeat until final answer  

---

## Example Flow

Query: "Who is the CEO of the company that owns Instagram?"

Thought: Instagram is owned by Meta  
Action: Search "Instagram owner"  
Observation: Meta  

Thought: Find Meta CEO  
Action: Search "Meta CEO"  
Observation: Mark Zuckerberg  

Final Answer: Mark Zuckerberg

In [25]:
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.tools import Tool
from langchain_core.prompts import PromptTemplate

In [34]:
google_search = GoogleSerperAPIWrapper()

tools = [
    Tool(
        name="WebAnswer",
        func=google_search.run,
        description="useful for when you need to ask with search",
        verbose=True
    )
]

In [35]:
## prompt template for ReAct agent

template = '''Answer the following questions as best you can. You have access to the following tools:
{tools}
Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
Begin!
Question: {input}
Thought:{agent_scratchpad}'''

In [36]:
prompt = PromptTemplate.from_template(template)

prompt.template

'Answer the following questions as best you can. You have access to the following tools:\n{tools}\nUse the following format:\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\nBegin!\nQuestion: {input}\nThought:{agent_scratchpad}'

In [37]:
## Creating agent

agent=create_agent(
    model=llm,
    tools=tools,
    system_prompt=prompt.template
)

In [38]:
response=agent.invoke({
    "messages":[
        {
            "role":"user",
            "content":"What is java"
        }
    ]
})

content='1. : coffee 2. usually Java : arabica coffee beans of plants grown in Java, Indonesia that produce a usually full-bodied coffee of low to medium acidity. Java is a high-level, general-purpose, memory-safe, object-oriented programming language. It is intended to let programmers write once, run anywhere (WORA). Java is a multi-platform, object-oriented, and network-centric language that can be used as a platform in itself. It is a fast, secure, reliable programming ... Java is a programming language and computing platform first released by Sun Microsystems in 1995. It has evolved from humble beginnings to power a large share ... Java is a high-level, object-oriented programming language developed by Sun Microsystems in 1995. It is mostly used for building desktop ... Java, modern object-oriented computer programming language. Java was created at Sun Microsystems, Inc., where James Gosling led a team of researchers. Java is a high-level, object-oriented programming language that 

In [39]:
print(response["messages"][-1].content)

Final Answer: Java is a high-level, general-purpose, memory-safe, object-oriented programming language that emphasizes simplicity, reliability, and portability. It is intended to let programmers write once, run anywhere (WORA) and is used for building desktop, mobile, web, and enterprise-level applications.


In [46]:
response=agent.invoke({
    "messages":[
        {
            "role":"user",
            "content":"Who is salman khan and how many movies he has done?"
        }
    ]
})

content='Salman Khan is an Indian actor and producer, known for his work in Hindi films. He made his film debut with a brief role in Biwi Ho To Aisi (1988) Explore the complete filmography of Salman Khan on Rotten Tomatoes! Discover every movie and TV show they have been credited in. Films starring Salman Khan · Poster for Om Shanti Om (2007) Om Shanti Om (2007) · Poster for Kuch Kuch Hota Hai (1998) · Poster for Pathaan (2023) · Poster for ... Salman Khan Films · 1. Jaanam Samjha Karo · 2. Har Dil Jo Pyar Karega... · 3. Chori Chori Chupke Chupke · 4. Dulhan Hum Le Jayenge · 5. Chal Mere Bhai · 6. Yeh ... Check out Salman Khan Box office collection till now. Also find out complete Salman Khan hit movie list. Also stay updated on Salman Khan latest videos, ... Shaadi Karke Phas Gaya Yaar Full Movie | Hindi Movies | Salman Khan Movies ... Suryavanshi Full Movie | Hindi Movies 2018 Full Movie | Salman Khan Movies | ... Salman Khan has 2 movies that I know of he has used Hanuman chalisa so

In [47]:
print(response["messages"][-1].content)

Final Answer: The final answer to the user's question is Salman Khan is an Indian actor and producer, known for his work in Hindi films. He made his film debut with a brief role in Biwi Ho To Aisi (1988). He has starred in more than eighty Hindi films.


### ReAct Agents with Custom Tools

In [89]:
from langchain.tools import tool

from langsmith import Client

client = Client()

## get the prompt template for a ReAct agent
prompt = client.pull_prompt("hwchase17/react")

prompt.template

'Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'

In [90]:
## Creating custom tools

@tool
def get_employee_id(name:str):
    """
    To get employee id, it takes employee name as arguments
    name(str): Name of the employee
    """
    fake_employees = {
    "Alice": "E001",
    "Bob": "E002",
    "Charlie": "E003",
    "Diana": "E004",
    "Evan": "E005",
    "Fiona": "E006",
    "George": "E007",
    "Hannah": "E008",
    "Ian": "E009",
    "Jasmine": "E010"}
  
    return fake_employees.get(name,"Employee not found")

In [91]:
@tool
def get_employee_salary(employee_id):
  """
  To get the salary of an employee, it takes employee_id as input and return salary
  """
  employee_salaries = {
    "E001": 56000,
    "E002": 47000,
    "E003": 52000,
    "E004": 61000,
    "E005": 45000,
    "E006": 58000,
    "E007": 49000,
    "E008": 53000,
    "E009": 50000,
    "E010": 55000
    }
  return employee_salaries.get(employee_id,"Employee not found")

In [92]:
tools=[get_employee_id,get_employee_salary]

In [93]:
## Creating agent with custom tools

agent=create_agent(
    model=llm,
    tools=tools,
    system_prompt=prompt.template
)

In [94]:
response=agent.invoke({
    "messages":[
        {
            "role":"user",
            "content":"What is the employee id for Evan and his salary use only tools to answer"
        }
    ]
})

In [ ]:
print(response["messages"][-1].content)

Final Answer: 45000
